# 01. GCR Baseline Reproduction

**Purpose:** Reproduce the Graph-Constrained Reasoning (GCR) baseline on WebQSP and CWQ.

**Output:** Saved predictions JSONL at `results/GenPaths/{dataset}/GCR-{model}/test/{config}/predictions.jsonl`

**Reference:** Luo et al. (2025) -- GCR achieves 92.6 Hits@1 on WebQSP, 75.8 on CWQ.

Based on: `workflow/predict_paths_and_answers.py` + `scripts/graph_constrained_decoding.sh`

**Note:** This is the unmodified GCR baseline, used to establish the reference SIR and Hits@1 for comparison with DCA-Trie variants in notebooks 02-04.

In [ ]:
# @title 1. Install & Setup
import sys
import os
import torch
import json
import math
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="transformers")

from tqdm import tqdm
from datasets import load_dataset
from multiprocessing import Pool
from functools import partial

# GPU detection with arch mapping
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
gpu_arch = (
    torch.cuda.get_device_properties(0).major * 10
    + torch.cuda.get_device_properties(0).minor
    if torch.cuda.is_available()
    else 0
)

ARCH_MAP = {"T4": 75, "L4": 89, "A100": 80, "H100": 90}
for _key, _arch in ARCH_MAP.items():
    if _key in gpu_name:
        gpu_arch = _arch
        break

has_a100 = "A100" in gpu_name
print(f"GPU: {gpu_name}  |  VRAM: {gpu_mem:.1f} GB  |  arch: sm_{gpu_arch}")

# Pinned dependencies (NO deepspeed/peft/bitsandbytes/trl/wandb/sentence-transformers)
!pip install -q transformers==4.44.2 "accelerate>=0.30.1" "datasets>=2.19.2" "marisa-trie>=1.2.0" "networkx>=3.0" "scikit-learn>=1.5.0" "tiktoken>=0.7.0" "sentencepiece>=0.2.0" "protobuf>=5.27.1"

# 3-tier flash-attn install: wheel -> source -> sdpa fallback
try:
    import flash_attn
    flash_attn_installed = True
except ImportError:
    flash_attn_installed = False
    print("Installing flash-attn (this may take several minutes)...")
    try:
        !pip install -q flash-attn --no-build-isolation
        import flash_attn
        flash_attn_installed = True
    except Exception:
        try:
            !pip install -q git+https://github.com/Dao-AILab/flash-attention.git
            import flash_attn
            flash_attn_installed = True
        except Exception:
            flash_attn_installed = False
            print("flash-attn not available, falling back to sdpa")

# Clone repo if needed
if not os.path.exists("graph-constrained-reasoning"):
    !git clone https://github.com/Adjanour/graph-constrained-reasoning-dca
%cd graph-constrained-reasoning
sys.path.insert(0, '.')
from src.llms import get_registed_model

In [ ]:
# @title 2. Configuration
import argparse
from src.qa_prompt_builder import PathGenerationWithAnswerPromptBuilder

# HF Token
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
except Exception:
    pass

# Google Drive integration (set DRIVE_BASE = None to use local paths)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = "/content/drive/MyDrive/GCR_Results"
    os.makedirs(DRIVE_BASE, exist_ok=True)
except Exception:
    DRIVE_BASE = None

# MODEL
MODEL_PATH = "rmanluo/GCR-Meta-Llama-3.1-8B-Instruct"
MODEL_NAME = "rmanluo/GCR-Meta-Llama-3.1-8B-Instruct"

# DATASETS
DATA_PATH = "rmanluo"
DATASETS = ["RoG-webqsp", "RoG-cwq"]
SPLIT = "test"

# DECODING
INDEX_LEN = 2
K = 5
GEN_MODE = "group-beam"
PROMPT_MODE = "zero-shot"
MAX_NEW_TOKENS = 256
N_PROC = 1
ATTN_IMPL = "flash_attention_2" if (has_a100 and flash_attn_installed) else "sdpa"

# SAMPLING
MAX_SAMPLES = 100          # set to None for full dataset

# OUTPUT
PREDICT_PATH = os.path.join(DRIVE_BASE, "GenPaths") if DRIVE_BASE else "results/GenPaths"
FORCE_RERUN = True

print(f"Model: {MODEL_PATH}")
print(f"Attention: {ATTN_IMPL}")
print(f"Datasets: {DATASETS}")
print(f"Beam width k: {K}")
print(f"Max path length: {INDEX_LEN}")
print(f"Max samples: {MAX_SAMPLES or 'full'}")

In [ ]:
# @title 3. Load Model (shared across datasets)
import argparse
from src.qa_prompt_builder import PathGenerationWithAnswerPromptBuilder

parser = argparse.ArgumentParser()
LLM = get_registed_model(MODEL_NAME)
LLM.add_args(parser)
args = parser.parse_args([])
args.model_path = MODEL_PATH
args.model_name = MODEL_NAME
args.k = K
args.generation_mode = GEN_MODE
args.attn_implementation = ATTN_IMPL
args.max_new_tokens = MAX_NEW_TOKENS
args.maximun_token = 4096

print("Loading model (this takes 1-2 minutes)...")
try:
    model = LLM(args)
    model.prepare_for_inference()
    # Force-clear sampling params to suppress warnings
    model.generation_cfg.temperature = None
    model.generation_cfg.top_p = None
    model.generation_cfg.top_k = None
    model.model.generation_config.temperature = None
    model.model.generation_config.top_p = None
    model.model.generation_config.top_k = None
    print("Model loaded.")
except Exception as e:
    print(f"Error loading model: {e}")
    raise

import src.utils as utils

In [ ]:
# @title 4. Run GCR Baseline on Selected Dataset
def get_output_file(path, force=False):
    """Open predictions file for writing, returning handle and list of already-processed IDs."""
    if not os.path.exists(path) or force:
        return open(path, "w"), []
    else:
        with open(path, "r") as f:
            processed = [json.loads(line)["id"] for line in f]
        fout = open(path, "a")
        return fout, processed


def predict_one(data, processed_list, input_builder, llm_model):
    """Generate a GCR prediction for a single question."""
    qid = data["id"]
    if qid in processed_list:
        return None
    input_query, ground_paths, trie = input_builder.process_input(data)
    if trie is None:
        return None
    start_token_ids = llm_model.tokenizer.convert_tokens_to_ids(input_builder.PATH_START_TOKEN)
    end_token_ids = llm_model.tokenizer.convert_tokens_to_ids(input_builder.PATH_END_TOKEN)
    llm_input = llm_model.prepare_model_prompt(input_query)
    prediction = llm_model.generate_sentence(
        llm_input, trie,
        start_token_ids=start_token_ids,
        end_token_ids=end_token_ids,
        enable_constrained_by_default=False,
    )
    if prediction is None:
        return None
    return {
        "id": qid,
        "question": data["question"],
        "prediction": prediction,
        "ground_truth": data["answer"],
        "ground_truth_paths": ground_paths,
        "input": input_query,
    }


SELECTED_DATASET = "RoG-webqsp"  # or "RoG-cwq"

print(f"Processing {SELECTED_DATASET}...")
dataset = load_dataset(f"{DATA_PATH}/{SELECTED_DATASET}", split=SPLIT)
print(f"Loaded {len(dataset)} questions")

if MAX_SAMPLES is not None:
    dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))
    print(f"Subsampled to {len(dataset)} questions")

input_builder = PathGenerationWithAnswerPromptBuilder(
    model.tokenizer, PROMPT_MODE, index_path_length=INDEX_LEN
)

postfix = f"{PROMPT_MODE}-{GEN_MODE}-k{K}-index_len{INDEX_LEN}"
data_name = SELECTED_DATASET
model_short = MODEL_PATH.split("/")[-1]
output_dir = os.path.join(PREDICT_PATH, data_name, model_short, SPLIT, postfix)
os.makedirs(output_dir, exist_ok=True)
print(f"Output: {output_dir}")

predictions_path = os.path.join(output_dir, "predictions.jsonl")
processed = []
if not FORCE_RERUN and os.path.exists(predictions_path):
    with open(predictions_path, "r") as f:
        processed = [json.loads(line)["id"] for line in f]
print(f"Already processed: {len(processed)} questions")

try:
    with open(predictions_path, "a" if processed else "w") as fout:
        for data in tqdm(dataset, desc="Generating"):
            res = predict_one(data, processed, input_builder, model)
            if res is not None:
                fout.write(json.dumps(res) + "\n")
                fout.flush()
except Exception as e:
    print(f"Error during generation: {e}")
    raise

from src.utils.qa_utils import eval_path_result_w_ans
print("\n=== Evaluation ===")
eval_path_result_w_ans(predictions_path)

In [ ]:
# @title 5. Compare with Published Results
print(
    """=== GCR Published Results (Luo et al. 2025) ===

                    WebQSP     CWQ
  GCR (Llama-3.1-8B)
    Hits@1           92.6      75.8
    F1               89.8      69.4
    Faithfulness    100%      100%

Your reproduction should be within 1-2% of these."""
)